# SIBA PyTorch 逐模块测试

本 Notebook 使用官方 PyTorch 源码和官方权重生成基准。受控张量只用于数值单元测试，不作为训练数据或论文实验结果。


## 1. 环境与路径


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path('/root/autodl-tmp/SIBA-Jittor')
TORCH_PYTHON = Path('/root/autodl-tmp/envs/PytorchDome/bin/python')
OFFICIAL_CHECKPOINT = PROJECT_ROOT / 'official_pytorch/checkpoint/SIBA_epoch60.pth'
TEST_ROOT = PROJECT_ROOT / 'logs/demo_module_tests'
PYTORCH_REFERENCE = TEST_ROOT / 'pytorch_seed2025.npz'
TEST_ROOT.mkdir(parents=True, exist_ok=True)
print('project:', PROJECT_ROOT)
print('checkpoint:', OFFICIAL_CHECKPOINT)


In [ ]:
!"{TORCH_PYTHON}" -c "import torch; print('PyTorch', torch.__version__); print('CUDA', torch.cuda.is_available())"


## 2. 官方源码与数据检查


In [ ]:
source_audit = json.loads((PROJECT_ROOT / 'docs/source_audit_final_20260728.json').read_text())
print(json.dumps(source_audit['comparison'], ensure_ascii=False, indent=2))
print((PROJECT_ROOT / 'logs/alignment/training_dataset_validation.json').read_text())
print((PROJECT_ROOT / 'logs/alignment/test_dataset_pairing.json').read_text())


## 3. 生成PyTorch基准


In [ ]:
!"{TORCH_PYTHON}" "{PROJECT_ROOT / 'tools/export_pytorch_alignment.py'}" --project-root "{PROJECT_ROOT}" --checkpoint "{OFFICIAL_CHECKPOINT}" --output "{PYTORCH_REFERENCE}" --batch-size 1 --height 32 --width 32 --seed 2025 --device cuda


In [ ]:
reference = np.load(PYTORCH_REFERENCE)
metadata = json.loads(PYTORCH_REFERENCE.with_suffix('.json').read_text())
print(metadata)
print('导出数组数量:', len(reference.files))

def show_arrays(names):
    rows = []
    for name in names:
        value = np.asarray(reference[name])
        rows.append({
            'name': name,
            'shape': str(value.shape),
            'min': float(value.min()),
            'max': float(value.max()),
            'mean': float(value.mean()),
            'std': float(value.std()),
        })
    return pd.DataFrame(rows)


## 4. 固定输入与权重


In [ ]:
display(show_arrays(['input_ir', 'input_vi']))
parameter_keys = [name for name in reference.files if name.startswith('parameter_initial__')]
print('参数张量数量:', len(parameter_keys))
print('示例参数:', parameter_keys[:5])


## 5. SE与Res-SE特征提取


In [ ]:
display(show_arrays(['activation__ir_conv', 'activation__vi_conv']))


## 6. LayerNorm、归一化与自注意力


In [ ]:
display(show_arrays(['activation__ir_sa_0', 'activation__vi_sa_0']))


## 7. CBSM源图权重


In [ ]:
display(show_arrays([
    'activation__weight_ir', 'activation__weight_irI',
    'activation__weight_vi', 'activation__weight_viI',
]))


## 8. 四路交叉注意力


In [ ]:
display(show_arrays([
    'activation__ir2vi_ca_0', 'activation__irI2vi_ca_0',
    'activation__vi2ir_ca_0', 'activation__viI2ir_ca_0',
]))


## 9. 特征拼接与输出


In [ ]:
display(show_arrays(['activation__mixed', 'activation__fuse_conv', 'activation__output']))


## 10. Laplacian、Intensity与Sobel损失


In [ ]:
loss_names = ['loss_laplacian', 'loss_intensity', 'loss_sobel', 'loss_total']
display(show_arrays(loss_names))
print('Loss = 10 × Laplacian + 0.1 × Intensity + Sobel')


## 11. 反向传播


In [ ]:
gradient_keys = [name for name in reference.files if name.startswith('gradient_preclip__')]
gradient_l2 = np.sqrt(sum(float((reference[name].astype(np.float64) ** 2).sum()) for name in gradient_keys))
print('梯度张量数量:', len(gradient_keys))
print('裁剪前全局L2范数:', gradient_l2)


## 12. 梯度裁剪


In [ ]:
postclip_keys = [name for name in reference.files if name.startswith('gradient_postclip__')]
postclip_l2 = np.sqrt(sum(float((reference[name].astype(np.float64) ** 2).sum()) for name in postclip_keys))
print('PyTorch返回的裁剪前范数:', float(reference['clip_total_norm']))
print('裁剪后全局L2范数:', postclip_l2)


## 13. Adam一步更新


In [ ]:
update_keys = [name for name in reference.files if name.startswith('parameter_update__')]
update_l2 = np.sqrt(sum(float((reference[name].astype(np.float64) ** 2).sum()) for name in update_keys))
print('参数更新张量数量:', len(update_keys))
print('一步更新全局L2范数:', update_l2)


## 14. PyTorch基准说明


In [ ]:
print('本Notebook输出是Jittor逐模块测试的参考值。')
print('最终实验指标仍来自完整60轮训练和706对测试，不来自32×32受控张量。')
